In [2]:
import librosa
import numpy as np
import os
from tqdm import tqdm
import glob
import pandas as pd
import numpy as np

def wav_to_mel(path, sr=16000, n_fft=1024, hop_length=256, n_mels=80):
    wav, _ = librosa.load(path, sr=sr)

    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=0,
        fmax=sr/2
    )

    log_mel = np.array(librosa.power_to_db(mel,ref=np.max))
    
    
    return log_mel



In [16]:
def feature_construction(root_path,n_mels=80,target_length=620,min_length=600):
    all_feature=[]
    file_info=[]  
    all_label=[]
    for style_file in os.listdir(root_path):
        style_file_path=os.path.join(root_path,style_file)
        if not os.path.isdir(style_file_path):
            continue
        song_files=os.listdir(style_file_path)
        for song_file in song_files:
            song_file_path=os.path.join(style_file_path,song_file)
            segment_file=glob.glob(os.path.join(song_file_path,"*.wav"))
            for segment in tqdm(segment_file,"segment_preprocessing"):
                try:
                    segment_feature = wav_to_mel(segment)
                    if segment_feature is None or np.any(np.isnan(segment_feature)):
                        print(f" {segment}: invalid!")
                        continue
                    if segment_feature.shape[1]<min_length:
                        continue
                    elif segment_feature.shape[1] < target_length:
                        padding = target_length - segment_feature.shape[1]
                        segment_feature = np.pad(segment_feature, 
                                               ((0, 0), (0, padding)), 
                                               mode='constant')
                    all_feature.append(segment_feature)  
                    segment_name = os.path.basename(segment)
                    file_info_temp = {
                        'style_name': style_file,
                        'song_name': song_file,
                        'segment_name': segment_name,
                        'segment_id': int(segment_name.split('_')[-1].split('.')[0])
                    }
                    
                    file_info.append(file_info_temp)
                    
                except Exception as e:
                    print(f"处理失败 {segment}: {e}")
                    continue
                file_info_temp={}
                all_label.append(0)                
    
    
    return all_label,all_feature,file_info
        

In [ ]:
print("=== 检查路径 ===")
print(f"根路径: {root_path}")
print(f"路径存在: {os.path.exists(root_path)}")

if os.path.exists(root_path):
    items = os.listdir(root_path)
    print(f"目录内容: {items}")

wav_files = glob.glob(os.path.join(root_path, "**", "*.wav"), recursive=True)
print(f"找到 {len(wav_files)} 个wav文件")

if wav_files:
    test_file = wav_files[0]
    print(f"\n=== 测试第一个文件 ===")
    print(f"测试文件: {test_file}")
    
    test_feature = wav_to_mel(test_file)
    if test_feature is not None:
        print(f"特征形状: {test_feature.shape}")
        print(f"NaN数量: {np.sum(np.isnan(test_feature))}")
    else:
        print("特征提取返回None")

print(f"\n=== 运行特征构建 ===")
final_df = feature_construction_debug(root_path)

if final_df is not None:
    print("🎉 成功！")
    print(final_df.head())
else:
    print("❌ 失败！")

In [17]:
root_path="/Users/robinhu/Desktop/project"
label_for_jazz,feature_jazz,info_jazz=feature_construction(root_path)


segment_preprocessing: 100%|█████████████████████| 7/7 [00:00<00:00, 128.95it/s]
segment_preprocessing: 0it [00:00, ?it/s]
segment_preprocessing: 100%|███████████████████| 11/11 [00:00<00:00, 127.84it/s]


In [38]:
feature_jazz_new=[]
for episode in feature_jazz:
    episode_temp=[]
    for mel in episode:
        mel=mel[:616]
        episode_temp.append(mel)
    feature_jazz_new.append(episode_temp)
    

In [65]:
info_list=[]
for episode in info_jazz:
    info_temp=[]
    info_names=episode.keys()
    for info_name in info_names:
        info_temp.append(episode[info_name])
    info_list.append(info_temp)


In [78]:
info_list_int=[]
for item in info_list:
    info_list_int.append([int(item[1].split('-')[-1]),item[-1]])

In [88]:
import torch
feature_jazz_tensor=torch.tensor(feature_jazz_new)
info_jazz_tensor=torch.tensor(info_list_int)
label_jazz_tensor=torch.tensor(label_for_jazz)
torch.save({'jazz_info':info_jazz_tensor,'jazz_feature':feature_jazz_tensor,'jazz_label':label_jazz_tensor},
           '/Users/robinhu/Desktop/torchdata/jazzdataset.pt')


In [114]:
def feature_construction2(root_path,n_mels=80,target_length=620,min_length=600,max_song=120):
    all_feature=[]
    file_info=[]  
    all_label=[]
    for style_file in os.listdir(root_path):
        style_file_path=os.path.join(root_path,style_file)
        if not os.path.isdir(style_file_path):
            continue
        song_files=os.listdir(style_file_path)[:max_song]
        for song_file in song_files:
            song_file_path=os.path.join(style_file_path,song_file)
            segment_file=glob.glob(os.path.join(song_file_path,"*.wav"))
            segment_num=0
            for segment in tqdm(segment_file,"segment_preprocessing"):
                if(segment_num>20):
                        break
                try:
                    segment_feature = wav_to_mel(segment)
                    if segment_feature is None or np.any(np.isnan(segment_feature)):
                        print(f" {segment}: invalid!")
                        continue
                    if segment_feature.shape[1]<min_length:
                        continue
                    elif segment_feature.shape[1] < target_length:
                        padding = target_length - segment_feature.shape[1]
                        segment_feature = np.pad(segment_feature, 
                                               ((0, 0), (0, padding)), 
                                               mode='constant')
                    all_feature.append(segment_feature)  
                    segment_name = os.path.basename(segment)
                    file_info.append({
                        'style_name': style_file,
                        'song_name': song_file,
                        'segment_name': segment_name,
                        'segment_id': int(segment_name.split('_')[-1].split('.')[0])
                    })
                    
                    segment_num+=1
                    all_label.append(1)
                    
                except Exception as e:
                    print(f"处理失败 {segment}: {e}")
                    continue
                
                
    return all_label,all_feature,file_info
        

In [115]:
root_path2="/Users/robinhu/Desktop/project2"
label_for_folk,feature_for_folk,info_for_folk=feature_construction2(root_path2)
zip_folk=[label_for_folk,feature_for_folk,info_for_folk]

segment_preprocessing:  29%|█████▍             | 22/77 [00:00<00:00, 111.96it/s]
segment_preprocessing: 0it [00:00, ?it/s]
segment_preprocessing: 100%|███████████████████| 19/19 [00:00<00:00, 114.62it/s]
/opt/anaconda3/envs/mert_python311/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=306
  warnings.warn(
segment_preprocessing:  55%|██████████▍        | 22/40 [00:00<00:00, 113.61it/s]
/opt/anaconda3/envs/mert_python311/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=300
  warnings.warn(
segment_preprocessing:  76%|██████████████▍    | 22/29 [00:00<00:00, 110.05it/s]


In [135]:
def create_tensor(data):
    import torch
    data_label=data[0]
    data_feature=data[1]
    data_info=data[2]
    feature_new=[]
    for episode in data_feature:
        episode_temp=[]
        for mel in episode:
            mel=mel[:616]
            episode_temp.append(mel)
        feature_new.append(episode_temp)
    info_list=[]
    for episode in data_info:
        info_temp=[]
        info_names=episode.keys()
        for info_name in info_names:
            info_temp.append(episode[info_name])
        info_list.append(info_temp)
    info_list_int=[]
    info_mapping={}
    count=0
    for item in info_list:
        if (item[1].split('-')[-1] not in info_mapping):
            count+=1
            info_mapping[count]=item[1].split('-')[-1]
        info_list_int.append([count,item[-1]])
    feature_tensor=torch.tensor(feature_new)
    info_tensor=torch.tensor(info_list_int)
    label_tensor=torch.tensor(data_label)
    return label_tensor,feature_tensor,info_tensor,info_mapping
        
        
    

In [136]:
folk_label_tensor,folk_feature_tensor,folk_info_tensor,folk_mapping=create_tensor(zip_folk)

In [138]:
torch.save({'folk_info':folk_info_tensor,'folk_feature':folk_feature_tensor,'folk_label':folk_label_tensor},
           '/Users/robinhu/Desktop/torchdata/folkdataset.pt')

In [141]:
folk=torch.load('/Users/robinhu/Desktop/torchdata/folkdataset.pt')
jazz=torch.load('/Users/robinhu/Desktop/torchdata/jazzdataset.pt')

In [146]:
info_total=torch.concat((folk['folk_info'],jazz['jazz_info']))
feature_total=torch.concat((folk['folk_feature'],jazz['jazz_feature']))
label_total=torch.concat((folk['folk_label'],jazz['jazz_label']))
torch.save({'info':info_total,'feature':feature_total,'label':label_total},'/Users/robinhu/Desktop/torchdata/totaldataset.pt')

In [152]:
torch.load('/Users/robinhu/Desktop/torchdata/jazz_transformation_total.pt')

tensor([[[-0.9915, -1.2962, -1.5699,  ..., -0.7783, -0.7572, -0.8520],
         [-1.2471, -1.5533, -1.8757,  ..., -0.1246, -0.2111, -0.3890],
         [-1.5811, -1.8869, -1.9688,  ...,  0.1185,  0.2268,  0.0448],
         ...,
         [ 0.1242,  0.5231,  0.5290,  ...,  0.2919,  0.2030,  0.2075],
         [ 0.1792,  0.3450,  0.4239,  ...,  0.7877,  0.7622,  0.6400],
         [ 0.0793,  0.3176,  0.3896,  ...,  0.7732,  0.8521,  0.8509]],

        [[-1.9252, -2.6440, -2.7338,  ..., -0.5653, -0.6707, -0.5738],
         [-2.0271, -2.6832, -2.7153,  ..., -0.3559, -0.4116, -0.5964],
         [-2.0734, -2.6975, -2.7770,  ...,  0.0613, -0.0254, -0.2020],
         ...,
         [-0.6363, -1.0965, -1.3308,  ...,  0.5467,  0.5372,  0.6042],
         [-0.6099, -1.0385, -1.2628,  ...,  0.8246,  0.7165,  0.7918],
         [-0.4806, -0.8064, -1.0103,  ...,  0.0719, -0.3542, -0.0206]],

        [[-0.0486, -0.3046, -0.2599,  ..., -0.3370, -0.3398, -0.4461],
         [ 0.1988, -0.1256, -0.1610,  ..., -0

In [151]:
folk

{'folk_info': tensor([[   1,   14],
         [   2,    0],
         [   3,   28],
         ...,
         [2205,    8],
         [2206,   20],
         [2207,   22]]),
 'folk_feature': tensor([[[-64.2813, -56.5932, -59.2113,  ..., -43.5829, -43.2522, -44.8887],
          [-67.4815, -60.4162, -63.6824,  ..., -29.2143, -29.8776, -33.8821],
          [-71.2561, -64.7515, -63.9419,  ..., -25.6069, -23.7843, -26.4539],
          ...,
          [-73.0796, -59.4156, -55.6978,  ..., -58.2842, -60.1825, -59.4139],
          [-73.2361, -62.7573, -58.0300,  ..., -53.9751, -54.1070, -54.9025],
          [-75.9863, -67.9689, -64.2375,  ..., -60.6188, -59.5905, -59.1787]],
 
         [[-80.0000, -80.0000, -80.0000,  ..., -40.4730, -42.4515, -40.9375],
          [-80.0000, -80.0000, -80.0000,  ..., -32.5597, -33.9275, -37.6074],
          [-80.0000, -80.0000, -80.0000,  ..., -26.0652, -28.0398, -30.1082],
          ...,
          [-80.0000, -80.0000, -80.0000,  ..., -55.3601, -55.5590, -53.8874],
    